# LEP-22 — Comparer plusieurs approches prédictives

## Objectif

Comparer plusieurs modèles pour estimer l’espérance de vie à la naissance
d’un pays à partir de ses indicateurs de la même année. La régression
linéaire multiple de LEP-21 sert de référence. Elle a obtenu une MAE de
1,95 an et une RMSE de 2,58 ans sur l’ensemble de validation.

## Protocole de comparaison

Nous entraînons les modèles sur les données de 2000 à 2011 et comparons
leurs performances sur les mêmes 374 observations de 2012 à 2013.
La MAE, exprimée en années, sert de critère principal. La RMSE complète
la comparaison en donnant davantage de poids aux erreurs importantes.

Une fois le modèle et ses réglages choisis, nous le réentraînons sur
les données de 2000 à 2013, puis nous l’évaluons sur les années 2014
et 2015, réservées au test final.

## 1. Préparer les données d’entraînement et de validation

In [2]:
# Bibliothèques nécessaires
from pathlib import Path

import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

from pretraitement_lep import imputer_par_statut

In [3]:
fichier = Path("../data/processed/life_expectancy_modelisation.csv")
df = pd.read_csv(fichier)

cible = "life_expectancy"
colonnes_exclues = {"country", "iso3", "year", "ensemble", cible}
variables_entree = [colonne for colonne in df.columns if colonne not in colonnes_exclues]
variables_numeriques = [colonne for colonne in variables_entree if colonne != "status"]

train = df.loc[df["ensemble"].eq("entrainement")]
val = df.loc[df["ensemble"].eq("validation")]

X_train, y_train = train[variables_entree], train[cible]
X_val, y_val = val[variables_entree], val[cible]

# Les médianes des deux ensembles proviennent de l'entraînement.
X_train_impute = imputer_par_statut(X_train, X_train, variables_numeriques)
X_val_impute = imputer_par_statut(X_train, X_val, variables_numeriques)

encodage = ColumnTransformer(
    [("statut", OneHotEncoder(drop="if_binary", handle_unknown="ignore"), ["status"])],
    remainder="passthrough",
)

## 2. Entraîner et comparer les deux modèles

In [4]:
estimateurs = {
    "Forêt aléatoire": RandomForestRegressor(random_state=42),
    "Gradient boosting": GradientBoostingRegressor(random_state=42),
}

# Scores de LEP-21 repris à deux décimales.
resultats = [
    {"modele": "Régression linéaire (LEP-21)", "mae": 1.95, "rmse": 2.58},
]

modeles_entraines = {}

for nom, estimateur in estimateurs.items():
    modele = make_pipeline(clone(encodage), estimateur)
    modele.fit(X_train_impute, y_train)
    predictions = modele.predict(X_val_impute)
    modeles_entraines[nom] = modele

    resultats.append({
        "modele": nom,
        "mae": mean_absolute_error(y_val, predictions),
        "rmse": mean_squared_error(y_val, predictions) ** 0.5,
    })

comparaison = (
    pd.DataFrame(resultats)
    .set_index("modele")
    .round(2)
    .sort_values("mae", kind="stable")
)

display(comparaison)

,mae,rmse
modele,,
Forêt aléatoire,1.05,1.73
Gradient boosting,1.28,1.84
Régression linéaire (LEP-21),1.95,2.58


La forêt aléatoire obtient les erreurs les plus faibles sur les 374 observations de validation de 2012–2013. En moyenne, ses estimations s’écartent de 1,05 an des valeurs observées, contre 1,28 an pour le gradient boosting et 1,95 an pour la régression linéaire de référence. Sa RMSE est également la plus faible (1,73 an), ce qui confirme son premier rang avec une mesure qui donne davantage de poids aux erreurs importantes. Nous retenons donc la forêt aléatoire avec les réglages évalués.




## 3.Réentraînement de la forêt aléatoire et évaluation finale

Nous réunissons les données d’entraînement et de validation pour réentraîner la forêt aléatoire sur l’ensemble de la période 2000–2013. Les médianes d’imputation sont recalculées sur cette période, puis appliquées aux données du test final. Nous évaluons ensuite le modèle sur les années 2014–2015 à l’aide de la MAE et de la RMSE, exprimées en années.

In [5]:
# Réunir l'entraînement et la validation, puis isoler le test final.
train_final = pd.concat([train, val], ignore_index=True)
test = df.loc[df["ensemble"].eq("test")]

X_train_final = train_final[variables_entree]
y_train_final = train_final[cible]
X_test = test[variables_entree]
y_test = test[cible]

# Calculer les médianes sur 2000–2013 et les appliquer aux deux ensembles.
X_train_final_impute = imputer_par_statut(
    X_train_final, X_train_final, variables_numeriques
)
X_test_impute = imputer_par_statut(
    X_train_final, X_test, variables_numeriques
)

# Reprendre les réglages évalués et réentraîner la forêt.
modele_final = clone(modeles_entraines["Forêt aléatoire"])
modele_final.fit(X_train_final_impute, y_train_final)

# Évaluer le modèle sur les observations de 2014–2015.
predictions_test = modele_final.predict(X_test_impute)
mae_test = mean_absolute_error(y_test, predictions_test)
rmse_test = mean_squared_error(y_test, predictions_test) ** 0.5

print(f"Observations de réentraînement : {len(y_train_final):,}".replace(",", " "))
print(f"Observations de test : {len(y_test)}")
print(f"MAE du test final : {mae_test:.2f} ans")
print(f"RMSE du test final : {rmse_test:.2f} ans")

Observations de réentraînement : 2 570
Observations de test : 366
MAE du test final : 0.88 ans
RMSE du test final : 1.31 ans


## Conclusion de LEP-22
Retenue pour ses résultats sur la validation, la forêt aléatoire a été réentraînée sur 2 570 observations de 2000 à 2013, puis évaluée sur 366 observations de 2014 à 2015.

Sur le test final, l’écart absolu entre l’espérance de vie estimée et la valeur observée est de 0,88 an en moyenne (MAE), soit environ 10,6 mois. La RMSE de 1,31 an donne davantage de poids aux erreurs les plus importantes.

Dans LEP-23, nous interpréterons les prédictions de la forêt aléatoire pour comprendre quels indicateurs contribuent le plus aux estimations et examiner les cas où le modèle se trompe le plus.

## Exporter le modèle et les prédictions

In [6]:
import joblib

dossier_modeles = Path("../models")
dossier_tables = Path("../reports/tables")
dossier_modeles.mkdir(parents=True, exist_ok=True)
dossier_tables.mkdir(parents=True, exist_ok=True)

joblib.dump(modele_final, dossier_modeles / "lep22_foret_aleatoire.joblib")

predictions_finales = test[["iso3", "country", "year", cible]].copy()
predictions_finales = predictions_finales.rename(columns={cible: "valeur_observee"})
predictions_finales["prediction"] = predictions_test
predictions_finales["erreur"] = (
    predictions_finales["prediction"] - predictions_finales["valeur_observee"]
)
predictions_finales["erreur_absolue"] = predictions_finales["erreur"].abs()

predictions_finales.to_csv(
    dossier_tables / "lep22_predictions_test.csv",
    index=False,
    encoding="utf-8-sig",
)

a supprimer 

a supprimer 

a supprimer 

a supprimer 

a supprimer 

a supprimer 

a supprimer 

a supprimer 

a supprimer 

a supprimer 

a supprimer 

a supprimer 

a supprimer 

a supprimer 